In [1]:
# !pip install pandas numpy matplotlib seaborn scikit-learn spacy
# !python -m spacy download en_core_web_sm

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece

### Kaggle: Name That LLM

In [2]:
import re
import math
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score
warnings.filterwarnings("ignore")

In [3]:
KAGGLE_DIR = Path("/kaggle/input/competitions/name-that-llm")
LOCAL_DIR = Path(".")

DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

VALID_MODELS = [
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "qwen3.8-max",
    "glm-5.3",
    "glm-5.3-flash",
    "kimi-k3",
]

N_SPLITS = 5
RANDOM_STATE = 42
TTR_WINDOW = 50

In [4]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

assert set(train["model"].unique()).issubset(set(VALID_MODELS))
assert train["id"].is_unique
assert test["id"].is_unique

print(train.shape, test.shape)
print(train["model"].value_counts().sort_index())
train.head()


(2699, 4) (3, 2)
model
claude-fable-5            149
claude-opus-4-6           150
claude-opus-4-8           150
claude-opus-5             150
claude-sonnet-5           150
gemini-3.1-pro-preview    150
gemini-3.6-flash          150
gemini-3.7-flash          150
gemini-3.8-flash          150
glm-5.3                   150
glm-5.3-flash             150
gpt-5.5                   150
gpt-5.6-luna              150
gpt-5.6-sol               150
gpt-5.6-terra             150
gpt-6-astra               150
kimi-k3                   150
qwen3.8-max               150
Name: count, dtype: int64


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


## Feature extraction


In [5]:
FUNCTION_WORDS = {
    "the", "a", "an", "and", "or", "but", "if", "while", "as", "of", "at", "by",
    "for", "with", "about", "against", "between", "into", "through", "during",
    "before", "after", "above", "below", "to", "from", "up", "down", "in", "out",
    "on", "off", "over", "under", "again", "further", "then", "once", "here",
    "there", "when", "where", "why", "how", "all", "any", "both", "each", "few",
    "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own",
    "same", "so", "than", "too", "very", "can", "will", "just", "should", "now",
    "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "this", "that", "these", "those", "it", "its",
}

FIRST_PERSON = {"i", "me", "my", "mine", "we", "us", "our", "ours", "myself", "ourselves"}

MODAL_VERBS = {
    "can", "could", "may", "might", "must", "shall", "should", "will", "would", "ought",
}

HEDGING = {
    "maybe", "perhaps", "possibly", "probably", "likely", "unlikely", "seem", "seems",
    "seemed", "appear", "appears", "appeared", "suggest", "suggests", "suggested",
    "approximately", "roughly", "somewhat", "relatively", "fairly", "quite", "rather",
    "arguably", "presumably", "supposedly", "apparently", "tend", "tends",
    "kind of", "sort of", "a bit", "almost", "nearly",
}

TRANSITIONS = {
    "however", "therefore", "moreover", "furthermore", "meanwhile", "consequently",
    "nevertheless", "nonetheless", "additionally", "similarly", "conversely",
    "hence", "thus", "instead", "otherwise", "accordingly",
    "in addition", "for example", "for instance", "in contrast", "on the other hand",
    "as a result", "in conclusion", "in summary", "that said",
}

CONTRACTIONS = re.compile(
    r"\b(?:[A-Za-z]+n't|[A-Za-z]+'(?:s|re|ve|ll|d|m))\b"
)

WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
SENT_RE = re.compile(r"[^.!?]+[.!?]+|[^.!?]+$")


In [6]:
def tokenize_words(text: str):
    return WORD_RE.findall(text.lower())


def split_sentences(text: str):
    parts = [s.strip() for s in SENT_RE.findall(text) if s.strip()]
    return parts if parts else ([text.strip()] if text.strip() else [])


def split_paragraphs(text: str):
    parts = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(parts) <= 1:
        parts = [p.strip() for p in text.split("\n") if p.strip()]
    return parts if parts else ([text.strip()] if text.strip() else [])


def count_syllables(word: str) -> int:
    word = word.lower()
    word = re.sub(r"[^a-z]", "", word)
    if not word:
        return 1
    if len(word) <= 3:
        return 1
    vowels = "aeiouy"
    count = 0
    prev_vowel = False
    for ch in word:
        is_vowel = ch in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel
    if word.endswith("e") and count > 1:
        count -= 1
    return max(count, 1)


def safe_div(a, b):
    return float(a) / float(b) if b else 0.0


def entropy_from_counts(counts):
    total = sum(counts.values())
    if total <= 0:
        return 0.0
    ent = 0.0
    for c in counts.values():
        p = c / total
        if p > 0:
            ent -= p * math.log2(p)
    return ent


def mtld(tokens, threshold=0.72):
    def _one_pass(seq):
        if not seq:
            return 0.0
        factors = 0.0
        types = set()
        token_count = 0
        for tok in seq:
            token_count += 1
            types.add(tok)
            ttr = len(types) / token_count
            if ttr <= threshold:
                factors += 1.0
                types = set()
                token_count = 0
        if token_count > 0:
            ttr = len(types) / token_count
            factors += (1.0 - ttr) / (1.0 - threshold) if threshold < 1 else 0.0
        return len(seq) / factors if factors else 0.0

    if len(tokens) < 10:
        return 0.0
    return 0.5 * (_one_pass(tokens) + _one_pass(list(reversed(tokens))))


def standardized_ttr(tokens, window=TTR_WINDOW):
    if not tokens:
        return 0.0
    if len(tokens) <= window:
        return safe_div(len(set(tokens)), len(tokens))
    scores = []
    for i in range(0, len(tokens) - window + 1, window):
        chunk = tokens[i : i + window]
        scores.append(safe_div(len(set(chunk)), len(chunk)))
    if not scores:
        chunk = tokens[:window]
        scores.append(safe_div(len(set(chunk)), len(chunk)))
    return float(np.mean(scores))


In [7]:
_nlp = None


def get_nlp():
    global _nlp
    if _nlp is not None:
        return _nlp
    try:
        import spacy
        _nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    except Exception:
        _nlp = False
    return _nlp


def syntactic_features(text: str, words: list):
    nlp = get_nlp()
    n_words = max(len(words), 1)
    out = {
        "noun_ratio": 0.0,
        "adjective_ratio": 0.0,
        "adverb_ratio": 0.0,
        "avg_dependency_distance": 0.0,
        "lexical_density": 0.0,
    }
    if not nlp or not text.strip():
        contentish = sum(1 for w in words if w not in FUNCTION_WORDS)
        out["lexical_density"] = safe_div(contentish, n_words)
        return out

    doc = nlp(text)
    tokens = [t for t in doc if not t.is_space and not t.is_punct]
    if not tokens:
        return out

    pos = [t.pos_ for t in tokens]
    n = len(pos)
    out["noun_ratio"] = safe_div(sum(p == "NOUN" or p == "PROPN" for p in pos), n)
    out["adjective_ratio"] = safe_div(sum(p == "ADJ" for p in pos), n)
    out["adverb_ratio"] = safe_div(sum(p == "ADV" for p in pos), n)
    content = sum(p in {"NOUN", "PROPN", "VERB", "ADJ", "ADV"} for p in pos)
    out["lexical_density"] = safe_div(content, n)

    dists = []
    for t in tokens:
        if t.head is not None and t.i != t.head.i:
            dists.append(abs(t.i - t.head.i))
    out["avg_dependency_distance"] = float(np.mean(dists)) if dists else 0.0
    return out


In [8]:
FEATURE_NAMES = [
    "char_count",
    "word_count",
    "sentence_count",
    "paragraph_count",
    "avg_sentence_length_words",
    "median_sentence_length_words",
    "std_sentence_length_words",
    "sentence_length_cv",
    "avg_word_length_chars",
    "avg_paragraph_length_words",
    "type_token_ratio",
    "hapax_legomena_ratio",
    "lexical_density",
    "lexical_diversity_mtld",
    "vocabulary_entropy",
    "short_word_ratio",
    "long_word_ratio",
    "function_word_frequency",
    "first_person_pronoun_frequency",
    "modal_verb_frequency",
    "hedging_word_frequency",
    "transition_word_frequency",
    "contraction_ratio",
    "repeated_bigram_ratio",
    "bigram_entropy",
    "comma_frequency",
    "semicolon_frequency",
    "colon_frequency",
    "em_dash_frequency",
    "question_mark_frequency",
    "punctuation_density",
    "multiple_punctuation_ratio",
    "markdown_heading_ratio",
    "bullet_item_count",
    "formatting_marker_density",
    "noun_ratio",
    "adjective_ratio",
    "adverb_ratio",
    "avg_dependency_distance",
    "flesch_reading_ease",
]


def extract_features(text: str) -> dict:
    text = text if isinstance(text, str) else ""
    words = tokenize_words(text)
    sentences = split_sentences(text)
    paragraphs = split_paragraphs(text)
    n_chars = len(text)
    n_words = len(words)
    n_sents = len(sentences)
    n_paras = len(paragraphs)

    sent_lens = [len(tokenize_words(s)) for s in sentences] or [0]
    avg_sent = float(np.mean(sent_lens))
    med_sent = float(np.median(sent_lens))
    std_sent = float(np.std(sent_lens)) if len(sent_lens) > 1 else 0.0
    cv_sent = safe_div(std_sent, avg_sent)

    word_lens = [len(w) for w in words] or [0]
    avg_word_len = float(np.mean(word_lens))

    para_word_counts = [len(tokenize_words(p)) for p in paragraphs] or [0]
    avg_para = float(np.mean(para_word_counts))

    counts = Counter(words)
    hapax = sum(1 for c in counts.values() if c == 1)
    # ttr = standardized_ttr(words, window=TTR_WINDOW)
    ttr = safe_div(len(set(words)), len(words))
    hapax_ratio = safe_div(hapax, n_words)
    vocab_ent = entropy_from_counts(counts)
    mtld_score = mtld(words)

    short_ratio = safe_div(sum(1 for w in words if len(w) <= 3), n_words)
    long_ratio = safe_div(sum(1 for w in words if len(w) >= 7), n_words)

    fw = safe_div(sum(1 for w in words if w in FUNCTION_WORDS), n_words)
    fp = safe_div(sum(1 for w in words if w in FIRST_PERSON), n_words)
    modal = safe_div(sum(1 for w in words if w in MODAL_VERBS), n_words)

    lower_text = text.lower()
    hedge_hits = 0
    for h in HEDGING:
        if " " in h:
            hedge_hits += len(re.findall(r"\b" + re.escape(h) + r"\b", lower_text))
        else:
            hedge_hits += sum(1 for w in words if w == h)
    hedge = safe_div(hedge_hits, n_words)

    trans_hits = 0
    for t in TRANSITIONS:
        if " " in t:
            trans_hits += len(re.findall(r"\b" + re.escape(t) + r"\b", lower_text))
        else:
            trans_hits += sum(1 for w in words if w == t)
    trans = safe_div(trans_hits, n_words)

    contractions = len(CONTRACTIONS.findall(text))
    contraction_ratio = safe_div(contractions, n_words)

    bigram_counts = Counter(zip(words, words[1:]))
    total_bigrams = sum(bigram_counts.values())

    repeated_bigram_ratio = safe_div(
        sum(c for c in bigram_counts.values() if c > 1),
        total_bigrams,
    )

    bigram_ent = entropy_from_counts(bigram_counts)

    denom = max(n_words, 1)
    comma_f = text.count(",") / denom
    semi_f = text.count(";") / denom
    colon_f = text.count(":") / denom
    em_dash_f = (text.count("—") + text.count("–") + text.count(" - ") + text.count("--")) / denom
    qmark_f = text.count("?") / denom
    punct_chars = sum(1 for ch in text if ch in ".,;:!?\"'()[]{}—–-")
    punct_density = safe_div(punct_chars, n_chars)
    multi_punct = len(re.findall(r"[!?]{2,}|\.{3,}", text))
    multi_punct_ratio = safe_div(multi_punct, max(n_sents, 1))

    lines = text.splitlines() or [text]
    heading_lines = sum(1 for ln in lines if re.match(r"^#{1,6}\s", ln.strip()))
    markdown_heading_ratio = safe_div(heading_lines, max(len(lines), 1))
    bullet_item_count = sum(
        1 for ln in lines if re.match(r"^(\s*[-*+]\s+|\s*\d+[.)]\s+)", ln)
    )
    fmt_markers = len(re.findall(r"(\*\*|__|`|~~|\*|\[)", text))
    formatting_marker_density = safe_div(fmt_markers, max(n_chars, 1))

    syn = syntactic_features(text, words)

    total_syllables = sum(count_syllables(w) for w in words) if words else 0
    flesch = (
        206.835
        - 1.015 * safe_div(n_words, max(n_sents, 1))
        - 84.6 * safe_div(total_syllables, max(n_words, 1))
    )

    feats = {
        "char_count": float(n_chars),
        "word_count": float(n_words),
        "sentence_count": float(n_sents),
        "paragraph_count": float(n_paras),
        "avg_sentence_length_words": avg_sent,
        "median_sentence_length_words": med_sent,
        "std_sentence_length_words": std_sent,
        "sentence_length_cv": cv_sent,
        "avg_word_length_chars": avg_word_len,
        "avg_paragraph_length_words": avg_para,
        "type_token_ratio": float(ttr),
        "hapax_legomena_ratio": hapax_ratio,
        "lexical_density": syn["lexical_density"],
        "lexical_diversity_mtld": float(mtld_score),
        "vocabulary_entropy": float(vocab_ent),
        "short_word_ratio": short_ratio,
        "long_word_ratio": long_ratio,
        "function_word_frequency": fw,
        "first_person_pronoun_frequency": fp,
        "modal_verb_frequency": modal,
        "hedging_word_frequency": hedge,
        "transition_word_frequency": trans,
        "contraction_ratio": contraction_ratio,
        "repeated_bigram_ratio": repeated_bigram_ratio,
        "bigram_entropy": float(bigram_ent),
        "comma_frequency": float(comma_f),
        "semicolon_frequency": float(semi_f),
        "colon_frequency": float(colon_f),
        "em_dash_frequency": float(em_dash_f),
        "question_mark_frequency": float(qmark_f),
        "punctuation_density": float(punct_density),
        "multiple_punctuation_ratio": float(multi_punct_ratio),
        "markdown_heading_ratio": float(markdown_heading_ratio),
        "bullet_item_count": float(bullet_item_count),
        "formatting_marker_density": float(formatting_marker_density),
        "noun_ratio": syn["noun_ratio"],
        "adjective_ratio": syn["adjective_ratio"],
        "adverb_ratio": syn["adverb_ratio"],
        "avg_dependency_distance": syn["avg_dependency_distance"],
        "flesch_reading_ease": float(flesch),
    }
    return feats


def build_feature_matrix(texts):
    rows = [extract_features(t) for t in texts]
    return pd.DataFrame(rows)[FEATURE_NAMES]

class HandcraftedTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return build_feature_matrix(list(X)).to_numpy(dtype=float)


In [9]:
_ = get_nlp()
print("spaCy:", "enabled" if _ else "fallback heuristics")

X_train = build_feature_matrix(train["text"].tolist())
X_test = build_feature_matrix(test["text"].tolist())

y_raw = train["model"].values
groups = train["prompt_id"].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

print(X_train.shape, X_test.shape)
X_train.head()


spaCy: enabled
(2699, 40) (3, 40)


,char_count,word_count,sentence_count,paragraph_count,avg_sentence_length_words,median_sentence_length_words,std_sentence_length_words,sentence_length_cv,avg_word_length_chars,avg_paragraph_length_words,...,punctuation_density,multiple_punctuation_ratio,markdown_heading_ratio,bullet_item_count,formatting_marker_density,noun_ratio,adjective_ratio,adverb_ratio,avg_dependency_distance,flesch_reading_ease
0,1143.0,193.0,8.0,1.0,24.125000,27.0,9.636357,0.399434,4.782383,193.0,...,0.024497,0.0,0.0,0.0,0.0,0.326425,0.093264,0.020725,2.405405,46.462115
1,518.0,67.0,4.0,1.0,16.750000,16.0,4.023369,0.240201,6.611940,67.0,...,0.021236,0.0,0.0,0.0,0.0,0.358209,0.074627,0.059701,2.047619,18.108377
2,474.0,77.0,4.0,1.0,19.250000,20.0,4.205651,0.218475,5.012987,77.0,...,0.025316,0.0,0.0,0.0,0.0,0.285714,0.233766,0.012987,2.273973,51.057289
3,1724.0,222.0,9.0,1.0,24.666667,24.0,4.216370,0.170934,6.576577,222.0,...,0.026102,0.0,0.0,0.0,0.0,0.391892,0.139640,0.063063,2.798122,2.309144
4,541.0,99.0,4.0,1.0,24.750000,26.0,2.861381,0.115611,4.343434,99.0,...,0.027726,0.0,0.0,0.0,0.0,0.262626,0.111111,0.040404,2.768421,66.350114


## Model


In [11]:
features = FeatureUnion([
    (
        "char",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            max_features=150_000,
            sublinear_tf=True,
        ),
    ),
    (
        "stats",
        Pipeline([
            ("extract", HandcraftedTransformer()),
            ("scale", StandardScaler()),
        ]),
    )
])
# (
#         "word",
#         TfidfVectorizer(
#             analyzer="word",
#             ngram_range=(1, 2),
#             min_df=2,
#             max_features=100_000,
#             sublinear_tf=True,
#         ),
#     )

clf = Pipeline([
    ("features", features),
    ("model", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

## Cross-validation evaluation


In [12]:
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_balanced_scores = []
fold_macro_f1_scores = []
oof_pred = np.empty(len(y), dtype=int)

for fold, (tr_idx, va_idx) in enumerate(
    sgkf.split(train["text"], y, groups),
    start=1,
):
    X_tr = train["text"].iloc[tr_idx].fillna("").astype(str)
    X_va = train["text"].iloc[va_idx].fillna("").astype(str)

    clf.fit(X_tr, y[tr_idx])
    pred = clf.predict(X_va)

    oof_pred[va_idx] = pred

    balanced_acc = balanced_accuracy_score(
        y[va_idx], pred
    )
    macro_f1 = f1_score(
        y[va_idx], pred, average="macro"
    )

    fold_balanced_scores.append(balanced_acc)
    fold_macro_f1_scores.append(macro_f1)

    print(
        f"Fold {fold}: "
        f"Balanced Accuracy={balanced_acc:.4f}, "
        f"Macro-F1={macro_f1:.4f}"
    )

cv_mean = float(np.mean(fold_balanced_scores))
cv_std = float(np.std(fold_balanced_scores))

print(
    f"\nMean balanced accuracy: {cv_mean:.4f} "
    f"+/- {cv_std:.4f}"
)
print(
    f"Mean macro-F1: "
    f"{np.mean(fold_macro_f1_scores):.4f}"
)

Fold 1: Balanced Accuracy=0.3503, Macro-F1=0.3323
Fold 2: Balanced Accuracy=0.3648, Macro-F1=0.3489
Fold 3: Balanced Accuracy=0.3481, Macro-F1=0.3324


KeyboardInterrupt: 

In [ ]:
oof_true_labels = le.inverse_transform(y)
oof_pred_labels = le.inverse_transform(oof_pred)

print(classification_report(oof_true_labels, oof_pred_labels, digits=3))
print("OOF balanced accuracy:", balanced_accuracy_score(y, oof_pred))


In [ ]:
cm = confusion_matrix(oof_true_labels, oof_pred_labels, labels=VALID_MODELS)
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_norm,
    xticklabels=VALID_MODELS,
    yticklabels=VALID_MODELS,
    cmap="Blues",
    vmin=0,
    vmax=1,
    square=True,
)
plt.title("OOF confusion matrix (row-normalized recall)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
per_class_recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
recall_df = pd.DataFrame(
    {"model": VALID_MODELS, "recall": per_class_recall}
).sort_values("recall")

plt.figure(figsize=(10, 5))
plt.barh(recall_df["model"], recall_df["recall"])
plt.xlabel("Recall")
plt.title("Per-class recall (OOF)")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

recall_df


## Fit full data and submit


In [ ]:
X_full = train["text"].fillna("").astype(str)
X_hidden = test["text"].fillna("").astype(str)

clf.fit(X_full, y)

test_pred = le.inverse_transform(
    clf.predict(X_hidden)
)

submission = pd.DataFrame({
    "id": test["id"],
    "model": test_pred,
})

assert submission["id"].is_unique
assert set(submission["model"]).issubset(set(VALID_MODELS))
assert len(submission) == len(test)

submission.to_parquet(
    "submission.parquet",
    index=False,
)

print(submission.shape)
print(submission.head())

## Quick sanity checks


In [ ]:
print("Train rows:", len(train))
print("Test rows:", len(test))
print("Features:", len(FEATURE_NAMES))
print("CV mean balanced accuracy:", round(cv_mean, 4))
print("Chance baseline:", round(1 / 18, 4))
print("Submission written: submission.parquet")
print(submission["model"].value_counts())
